# Extract Lists for Towns, Cities, Hamlets and Villages 

From OS Data Hub (Open Names) dataset, https://osdatahub.os.uk/data/downloads/open/OpenNames.

In [1]:
import geopandas as gpd

file = gpd.read_file(r"C:\Users\harrid3\Downloads\opname_gpkg_gb\Data\opname_gb.gpkg")

In [ ]:
file.columns

Towns, Cities, Hamlets, Bays and Suburban Areas can be found within the `local_type` column of the Open Names dataset. These are processed to uppercase, printed (may need to use pprint in the case of longer lists), and formatted within NotePad++ to be copied over to the `address_cleaning.resources` folder.

The stages of processing in NotePad++ are as follows:
- `Ctrl+H` to bring up processing window click on the `Replace` tab
- Replace all instances of `\r\n` with an empty string (this places all items to one line)
- Replace all instances of `(([^,]+,\s*){6}[^,]+)` with `\1\r\n` (this places every 6 items to a new line)
- Copy output over to `address_cleaning.resources`

In [ ]:
towns = file[file['local_type']=='Town']['name1'].unique().tolist()
cities = file[file['local_type']=='City']['name1'].unique().tolist()
villages = file[file['local_type']=='Village']['name1'].unique().tolist()
hamlets = file[file['local_type']=='Hamlet']['name1'].unique().tolist()
suburbs = file[file['local_type']=='Suburban Area']['name1'].unique().tolist()
bays = file[file['local_type']=='Bay']['name1'].unique().tolist()

In [ ]:
towns = [s.upper() for s in towns]
cities = [s.upper() for s in cities]
villages = [s.upper() for s in villages]
hamlets = [s.upper() for s in hamlets]
suburbs = [s.upper() for s in suburbs]
bays = [s.upper() for s in bays]

In [ ]:
from pprint import pprint

pprint(bays)

To extract boroughs/districts, counties and populated places. These are found as distinct column names.

In [ ]:
def split_string(s, delimiter):
    return [item.strip() for item in s.split(delimiter) if item.strip()]

In [ ]:
boroughs = file['district_borough'].unique().tolist()
boroughs = [s.upper() for s in boroughs if not isinstance(s, float) and s]
boroughs = [split_string(s, ' - ') for s in boroughs]
boroughs = [item for sublist in boroughs for item in sublist]

counties = file['county_unitary'].unique().tolist()
counties = [s.upper() for s in counties if not isinstance(s, float) and s]
counties = [split_string(s, ' - ') for s in counties]
counties = [item for sublist in counties for item in sublist]

populated_places = file['populated_place'].unique().tolist()
populated_places = [s.upper() for s in populated_places if not isinstance(s, float) and s]

Populated Places need specific processing as some include native language name and english name in one component.

In [ ]:
populated_places_no_process = [s for s in populated_places if "/" not in s]
populated_places_to_process = [s for s in populated_places if "/" in s]
populated_places_to_process = [s.split("/") for s in populated_places_to_process]
populated_places_processed = [s.strip() for sublist in populated_places_to_process for s in sublist]

populated_places = populated_places_no_process + populated_places_processed

In [ ]:

pprint(counties)

Generating lists to be used in contextualise address functionality

In [14]:
towns_districts_counties = file[file['local_type']=='Town'][['name1','district_borough','county_unitary','postcode_district']]
town_lookup = {}
for index, row in towns_districts_counties.iterrows():

    town = row['name1'] if not isinstance(row['name1'], float) and row['name1'] else ''
    district = row['district_borough'] if not isinstance(row['district_borough'], float) and row['district_borough'] else ''
    county = row['county_unitary'] if not isinstance(row['county_unitary'], float) and row['county_unitary'] else ''
    postcode = row['postcode_district'] if not isinstance(row['postcode_district'], float) and row['postcode_district'] else ''

    hold = [town, district, county, postcode]
    lookup = list(dict.fromkeys(hold))

    town_lookup[town.upper()] = lookup

In [ ]:
city_counties = file[file['local_type']=='City'][['name1','county_unitary']]
city_lookup = {}
for index, row in city_counties.iterrows():

    city = row['name1'] if row['name1'] else ''
    county = row['county_unitary'] if row['county_unitary'] else ''

    hold = [city, county]
    lookup = list(dict.fromkeys(hold))

    city_lookup[city.upper()] = lookup

In [12]:
villages_district_counties = file[file['local_type']=='Village'][['name1','district_borough','county_unitary','postcode_district']]
village_lookup = {}
for index, row in villages_district_counties.iterrows():

    village = row['name1'] if row['name1'] else ''
    district = row['district_borough'] if row['district_borough'] else ''
    county = row['county_unitary'] if row['county_unitary'] else ''
    postcode = row['postcode_district'] if row['postcode_district'] else ''

    hold = [village,district, county, postcode]
    lookup = list(dict.fromkeys(hold))

    village_lookup[village.upper()] = lookup

In [15]:
pprint(town_lookup)

{'ABERAERON': ['Aberaeron', '', 'Sir Ceredigion - Ceredigion', 'SA46'],
 'ABERCARN': ['Abercarn', '', 'Caerffili - Caerphilly', 'NP11'],
 'ABERCHIRDER': ['Aberchirder', '', 'Aberdeenshire', 'AB54'],
 'ABERCYNON': ['Abercynon',
               '',
               'Rhondda Cynon Taf - Rhondda Cynon Taf',
               'CF45'],
 'ABERDARE': ['Aberdare', '', 'Rhondda Cynon Taf - Rhondda Cynon Taf', 'CF44'],
 'ABERDAUGLEDDAU': ['Aberdaugleddau', '', 'Sir Benfro - Pembrokeshire', 'SA73'],
 'ABERFELDY': ['Aberfeldy', '', 'Perth and Kinross', 'PH15'],
 'ABERGAVENNY': ['Abergavenny', '', 'Sir Fynwy - Monmouthshire', 'NP7'],
 'ABERGELE': ['Abergele', '', 'Conwy - Conwy', 'LL22'],
 'ABERGWAUN': ['Abergwaun', '', 'Sir Benfro - Pembrokeshire', 'SA65'],
 'ABERHONDDU': ['Aberhonddu', '', 'Powys - Powys', 'LD3'],
 'ABERMAW': ['Abermaw', '', 'Gwynedd - Gwynedd', 'LL42'],
 'ABERSYCHAN': ['Abersychan', '', 'Torfaen - Torfaen', 'NP4'],
 'ABERTEIFI': ['Aberteifi', '', 'Sir Ceredigion - Ceredigion', 'SA43'],

In [18]:
hamlets_district_counties = file[file['local_type']=='Hamlet'][['name1','district_borough','county_unitary','postcode_district']]
hamlet_lookup = {}
for index, row in hamlets_district_counties.iterrows():

    hamlet = row['name1'] if not isinstance(row['name1'], float) and row['name1'] else ''
    district = row['district_borough'] if not isinstance(row['district_borough'], float) and row['district_borough'] else ''
    county = row['county_unitary'] if not isinstance(row['county_unitary'], float) and row['county_unitary'] else ''
    postcode = row['postcode_district'] if not isinstance(row['postcode_district'], float) and row['postcode_district'] else ''


    hold = [hamlet,district, county, postcode]
    lookup = list(dict.fromkeys(hold))

    hamlet_lookup[hamlet.upper()] = lookup

In [19]:
pprint(hamlet_lookup)

{"A' CHILL": ["A' Chill", '', 'Highland', 'PH44'],
 'AB LENCH': ['Ab Lench', 'Wychavon', 'Worcestershire', 'WR11'],
 'ABATY CWM-HIR': ['Abaty Cwm-hir', '', 'Powys - Powys', 'LD1'],
 'ABBERTON': ['Abberton', 'Wychavon', 'Worcestershire', 'WR10'],
 'ABBEY': ['Abbey', 'East Devon', 'Devon', 'EX14'],
 'ABBEY GREEN': ['Abbey Green',
                 'Staffordshire Moorlands',
                 'Staffordshire',
                 'ST13'],
 'ABBEY ST BATHANS': ['Abbey St Bathans', '', 'Scottish Borders', 'TD11'],
 'ABBEYSTEAD': ['Abbeystead', 'Lancaster', 'Lancashire', 'LA2'],
 'ABBOTS BICKINGTON': ['Abbots Bickington', 'Torridge', 'Devon', 'EX22'],
 'ABBOTSFORD': ['Abbotsford', 'Mid Sussex', 'West Sussex', 'RH15'],
 'ABBOTSTONE': ['Abbotstone', 'Winchester', 'Hampshire', 'SO24'],
 'ABBOTSWOOD': ['Abbotswood', 'Wychavon', 'Worcestershire', 'WR5'],
 'ABCOTT': ['Abcott', '', 'Shropshire', 'SY7'],
 'ABDON': ['Abdon', '', 'Shropshire', 'SY7'],
 'ABER': ['Aber', '', 'Sir Ceredigion - Ceredigion', 'SA

In [20]:
suburbs_district_counties = file[file['local_type']=='Suburban Area'][['name1','district_borough','county_unitary','postcode_district']]
suburb_lookup = {}
for index, row in suburbs_district_counties.iterrows():

    suburb = row['name1'] if not isinstance(row['name1'], float) else ''
    district = row['district_borough'] if not isinstance(row['district_borough'], float) else ''
    county = row['county_unitary'] if not isinstance(row['county_unitary'], float) else ''
    postcode = row['postcode_district'] if not isinstance(row['postcode_district'], float) else ''


    hold = [suburb,district, county, postcode]
    lookup = list(dict.fromkeys(hold))

    suburb_lookup[suburb.upper()] = lookup

In [21]:
pprint(suburb_lookup)

{"AARON'S HILL": ["Aaron's Hill", 'Waverley', 'Surrey', 'GU7'],
 'ABBEY END': ['Abbey End', 'Warwick', 'Warwickshire', 'CV8'],
 'ABBEY FIELDS': ['Abbey Fields', 'Warwick', 'Warwickshire', 'CV8'],
 'ABBEY HULTON': ['Abbey Hulton', '', 'City of Stoke-on-Trent', 'ST2'],
 'ABBEY MEADS': ['Abbey Meads', '', 'Swindon', 'SN25'],
 'ABBEY PARKS': ['Abbey Parks', '', 'Fife', 'KY12'],
 'ABBEY WOOD': ['Abbey Wood', 'Greenwich', 'Greater London', 'SE2'],
 'ABBEYDALE': ['Abbeydale', 'Gloucester', 'Gloucestershire', 'GL4'],
 'ABBEYDALE PARK': ['Abbeydale Park', 'Sheffield', '', 'S17'],
 'ABBEYHILL': ['Abbeyhill', '', 'City of Edinburgh', 'EH7'],
 "ABBOT'S MEAD": ["Abbot's Mead", '', 'Cheshire West and Chester', 'CH1'],
 'ABBOTSBURY': ['Abbotsbury', 'Teignbridge', 'Devon', 'TQ12'],
 'ABBOTSMEAD': ['Abbotsmead', '', 'Westmorland and Furness', 'LA13'],
 'ABBOTSWOOD': ['Abbotswood', 'Guildford', 'Surrey', 'GU1'],
 'ABBOTTS BARTON': ['Abbotts Barton', 'Winchester', 'Hampshire', 'SO23'],
 'ABER PWLL': ['Ab

In [24]:
bays_district_counties = file[file['local_type']=='Bay'][['name1','district_borough','county_unitary','postcode_district']]
bay_lookup = {}
for index, row in bays_district_counties.iterrows():

    bay = row['name1'] if not isinstance(row['name1'], float) else ''
    district = row['district_borough'] if not isinstance(row['district_borough'], float) else ''
    county = row['county_unitary'] if not isinstance(row['county_unitary'], float) else ''
    postcode = row['postcode_district'] if not isinstance(row['postcode_district'], float) else ''

    hold = [bay,district, county, postcode]
    lookup = list(dict.fromkeys(hold))

    bay_lookup[bay.upper()] = lookup

In [25]:
pprint(bay_lookup)

{"A' CHLÈIT": ["A' Chlèit", '', 'Highland', 'IV21'],
 'AAGLASS': ['Aaglass', '', 'Shetland Islands', 'ZE2'],
 'ABER BACH': ['Aber Bach', '', 'Sir Benfro - Pembrokeshire', 'SA65'],
 'ABER CAREG-Y-FRAN': ['Aber Careg-y-Fran',
                       '',
                       'Sir Benfro - Pembrokeshire',
                       'SA42'],
 'ABER CAWELL': ['Aber Cawell', '', 'Sir Ynys Mon - Isle of Anglesey', 'LL68'],
 'ABER CEFFYL': ['Aber Ceffyl', '', 'Sir Benfro - Pembrokeshire', 'SA62'],
 'ABER CERRIG-GWYNION': ['Aber Cerrig-gwynion',
                         '',
                         'Sir Benfro - Pembrokeshire',
                         'SA64'],
 'ABER CLAWDD PRIDD': ['Aber Clawdd Pridd',
                       '',
                       'Sir Benfro - Pembrokeshire',
                       'SA64'],
 'ABER DEGAN': ['Aber Degan', '', 'Sir Benfro - Pembrokeshire', 'SA64'],
 'ABER DRAW': ['Aber Draw', '', 'Sir Benfro - Pembrokeshire', 'SA62'],
 'ABER DWYRAIN': ['Aber Dwyrain', '', 'Sir 